- https://github.com/huggingface/transformers/tree/main/examples/pytorch/contrastive-image-text

### ViT

- 取代 cnn 在 CV 中的地位
    - cnn：局部感受些，比较难建立全局感知，transformer 的 attention 本身就是全局的；
    - transformer 比 cnn 调参容易，transformer 中的每个 layer 大体相同，cnn（resnet）多阶段（阶段内也需要大量手动调参）
- ViT就是纯Encoder结构，会让每个视觉Token都彼此看见。
    - 感受野 (Receptive Field)：这也是 ViT 与 CNN 最大的区别之一。CNN 的感受野通常随着层数加深逐渐变大，而 ViT 从第一层开始，其理论感受野就是全图（Global）。

### CLIP

- encoder
    - text encoder
    - visual encoder
        - ViT: global attention (ViT 的 T 顾名思义就是 transformer)

### SigLIP

> 将 CLIP 中的“多分类检索问题”转化为“成对的二分类问题”。

- LIP 本质上是在做Batch 内的多分类 (N-way Classification)。对于每一张图片，模型要在 $N$ 个文本中找出正确的那个。
    - 数学公式：对于第 $i$ 个样本对，CLIP 的 Image-to-Text 损失函数为：$$L_{CLIP}^{(i)} = -\log \frac{\exp(t \cdot x_i \cdot y_i)}{\sum_{j=1}^N \exp(t \cdot x_i \cdot y_j)}$$
    - Softmax 强制概率和为 1。这意味着如果模型认为图片 $i$ 和文本 $j$ (负样本) 很像，它必然会降低对正样本 $i$ 的预测概率。
- SigLIP 放弃了 Softmax，将每一个 Image-Text 对 $(i, j)$ 视为一个独立的二分类任务（匹配 vs 不匹配）。
    - 数学公式：定义 Logit 为 $z_{ij} = t \cdot x_i \cdot y_j + b$。我们引入标签 $l_{ij}$：当 $i=j$ (正样本) 时为 $1$，否则 (负样本) 为 $-1$。SigLIP 的损失函数是所有 $N \times N$ 个配对的 Logistic Loss 之和：$$L_{SigLIP} = - \frac{1}{N} \sum_{i=1}^N \sum_{j=1}^N \log \sigma(l_{ij} \cdot z_{ij})$$
    - 展开来看，就是正样本项和负样本项的加权和：$$L = - \frac{1}{N} \left[ \underbrace{\sum_{i=1}^N \log \sigma(t x_i y_i + b)}_{\text{正样本 (对角线)}} + \underbrace{\sum_{i=1}^N \sum_{j \neq i} \log (1 - \sigma(t x_i y_j + b))}_{\text{负样本 (非对角线)}} \right]$$
    - 它可以认为一张图同时匹配两个文本（Sigmoid 输出都是 High），这其实更符合真实世界语义（多义性）。

### simple code

In [1]:
import clip

In [4]:
clip.available_models()

['RN50',
 'RN101',
 'RN50x4',
 'RN50x16',
 'RN50x64',
 'ViT-B/32',
 'ViT-B/16',
 'ViT-L/14',
 'ViT-L/14@336px']

In [5]:
clip_version = "ViT-B/16"
device = 'cuda:6'
model, preprocess = clip.load(clip_version, device=device)  # clip.available_models()

100%|███████████████████████████████████████| 335M/335M [02:58<00:00, 1.97MiB/s]


In [6]:
model.eval()

clip_feat_dim = {'RN50': 1024, 'RN101': 512, 'RN50x4': 640, 'RN50x16': 768, 'RN50x64': 1024, 
                 'ViT-B/32': 512, 'ViT-B/16': 512, 'ViT-L/14': 768}[clip_version]

In [7]:
def get_text_feats(in_text, batch_size=64):
  text_tokens = clip.tokenize(in_text).to(device)
  text_id = 0
  text_feats = np.zeros((len(in_text), clip_feat_dim), dtype=np.float32)
  while text_id < len(text_tokens):  # Batched inference.
    batch_size = min(len(in_text) - text_id, batch_size)
    text_batch = text_tokens[text_id:text_id+batch_size]
    with torch.no_grad():
      batch_feats = model.encode_text(text_batch).float()
    batch_feats /= batch_feats.norm(dim=-1, keepdim=True)
    batch_feats = np.float32(batch_feats.cpu())
    text_feats[text_id:text_id+batch_size, :] = batch_feats
    text_id += batch_size
  return text_feats

def get_img_feats(img):
  img_pil = Image.fromarray(np.uint8(img))
  img_in = preprocess(img_pil)[None, ...]
  with torch.no_grad():
    img_feats = model.encode_image(img_in.to(device)).float()
  img_feats /= img_feats.norm(dim=-1, keepdim=True)
  img_feats = np.float32(img_feats.cpu())
  return img_feats

# "nearest neighbor"
def get_nn_text(raw_texts, text_feats, img_feats):
  scores = text_feats @ img_feats.T
  scores = scores.squeeze()
  high_to_low_ids = np.argsort(scores).squeeze()[::-1]
  high_to_low_texts = [raw_texts[i] for i in high_to_low_ids]
  high_to_low_scores = np.sort(scores).squeeze()[::-1]
  return high_to_low_texts, high_to_low_scores